In [14]:
import torch
from pathlib import Path
import os
import json
import pandas as pd
from tqdm import tqdm
from model import DiagnosticModel
from train_utils import get_info
from collections import defaultdict
import numpy as np

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

In [28]:
results_dir = Path('../outputs')
device = 'cuda'

In [42]:
results_dict = defaultdict(list)

for sbdir in tqdm(os.listdir(results_dir )):
    sbdir_path = results_dir / sbdir
    
    model_path = sbdir_path / 'best_model.pth'
    params_path = sbdir_path / 'params.json'
    results_path = sbdir_path / 'test_results.json'

    with open(params_path, 'r') as f:
        params = json.load(f)

    if not os.path.exists(results_path):
        continue 
    with open(results_path, 'r') as f:
        results = json.load(f)

    checkpoint = torch.load(model_path, map_location=device)
    # model = DiagnosticModel(model_name = params['model'], in_channels = 1)
    # model = model.to(device)
    # model.load_state_dict(checkpoint['model_state_dict'])
    val_metric = params['val_metric']
    start_epoch = checkpoint['epoch']
    # best_val_metric_score = checkpoint[val_metric]
    full_train = checkpoint['full_train']
    full_val = checkpoint['full_val']
    full_loss = checkpoint['full_loss']
    
    results_dict['exps'].append(sbdir)
    results_dict['auc'].append(float(results['auc']))
    # results_dict['val_metric'].append(float(best_val_metric_score)) # ignore b/c sometimes f1 and sometimes acc

    for test_metric, test_value in get_info(np.array(results['val_cfvalues'])).items():
        results_dict[f'test_{test_metric}'].append(test_value)
    


100%|██████████| 17/17 [00:03<00:00,  5.65it/s]


In [43]:
sort_metric = ['auc', 'test_acc']

df = pd.DataFrame(results_dict)
df = df[['exps'] + sorted([col for col in df.columns if col != 'exps'])]

df = df.sort_values(by=sort_metric, ascending=False)
df = df.reset_index(drop=True)
df


,exps,auc,test_acc,test_f1,test_fn,test_fp,test_fpr,test_prec,test_recall,test_tn,test_tp,test_tpr
0,b_mask2,0.850796,0.925134,0.382353,0.062389,0.012478,0.013645,0.650000,0.270833,0.901961,0.023173,0.270833
1,balance_o_download,0.840459,0.921569,0.368421,0.075163,0.003268,0.003623,0.875000,0.233333,0.898693,0.022876,0.233333
2,balance_b_download,0.835326,0.918301,0.500000,0.057190,0.024510,0.027174,0.625000,0.416667,0.877451,0.040850,0.416667
3,balance_b_download2,0.819746,0.919935,0.436782,0.066993,0.013072,0.014493,0.703704,0.316667,0.888889,0.031046,0.316667
4,b_stack2,0.814191,0.913399,0.495238,0.055556,0.031046,0.034420,0.577778,0.433333,0.870915,0.042484,0.433333
5,balance_o_download2,0.808137,0.924837,0.477273,0.063725,0.011438,0.012681,0.750000,0.350000,0.890523,0.034314,0.350000
6,balance_w_download,0.807820,0.918301,0.479167,0.060458,0.021242,0.023551,0.638889,0.383333,0.880719,0.037582,0.383333
7,b_mask,0.800073,0.926916,0.327869,0.067736,0.005348,0.005848,0.769231,0.208333,0.909091,0.017825,0.208333
8,b_stack,0.796407,0.929739,0.505747,0.062092,0.008170,0.009058,0.814815,0.366667,0.893791,0.035948,0.366667
9,balance_w,0.778019,0.872549,0.390625,0.057190,0.070261,0.077899,0.367647,0.416667,0.831699,0.040850,0.416667
